# Exercise 7.10 — A Richer QELM POVM from Two Reservoir Qubits

**Chapter 7: Quantum Machine Learning** &nbsp;|&nbsp; *Exercises*

---

Section 7.5 builds the smallest quantum extreme learning machine, one input qubit and one reservoir
qubit, and folds the reservoir into an effective four-outcome POVM on the input. The natural next move
is to enlarge the reservoir. Take *two* reservoir qubits in $\ket{00}$, a Hadamard on the input, and a
CNOT chain running through the reservoir. Eight outcomes are now available where there were four, so
the readout should be richer. It is not, and the reason is worth more than the counting.

## 1. Physical Motivation

A QELM never trains its quantum layer. A fixed unitary $\hat{U}_{\mathrm{res}}$ couples the encoded
input to a reservoir, the joint system is measured, and only the classical readout weights are fitted.
Everything the model can ever learn is therefore decided in advance by the measurement, and Section 7.5
makes this precise: tracing out the reservoir turns the whole pipeline into a POVM acting on the input
alone (Eq. 7.36),
$$
p_b(x) = \Tr_{\mathrm{in}}\bigl[\hat{E}_b\,\hat\rho_{\mathrm{in}}(x)\bigr] , \qquad
\hat{E}_b = \bra{0}_{\mathrm{res}}\hat{U}_{\mathrm{res}}^\dagger\ket{b}\!\bra{b}
\hat{U}_{\mathrm{res}}\ket{0}_{\mathrm{res}} .
$$
The span of $\{\hat{E}_b\}$ fixes the accessible information. A target function outside that span is
invisible to the model, and no amount of training data brings it back.

This is why counting outcomes is a trap. Reservoir qubits enlarge the Naimark dilation space and so
raise the *number* of available $\hat{E}_b$, but the useful quantity is how many linearly independent
operators they span. The circuit here separates the two.

## 2. Mathematical Toolbox

**The circuit.** Qubit $1$ carries the input, qubits $2$ and $3$ are the reservoir in $\ket{00}$, and
$$
\hat{V} = \mathrm{CNOT}_{23}\,\mathrm{CNOT}_{12}\,\hat{H}_1 .
$$

**Its action.** For an input $\ket{\psi} = \alpha\ket{+} + \beta\ket{-}$, the Hadamard sends
$\ket{\pm}\mapsto\ket{0},\ket{1}$, and the CNOT chain copies that bit down the register,
$$
\hat{V}\ket{\psi}\ket{00} = \alpha\ket{000} + \beta\ket{111} ,
\qquad \alpha = \braket{+|\psi}, \;\; \beta = \braket{-|\psi} .
$$
Six of the eight computational-basis strings never appear.

**The effects.** With $\hat{A}_b = \bra{b}\hat{V}\ket{00}_{\mathrm{res}}$ a row vector on the input
space, $\hat{E}_b = \hat{A}_b^\dagger\hat{A}_b$, manifestly positive and of rank at most one. Reading
off the two surviving amplitudes,
$$
p_{000} = |\alpha|^2 = \braket{+|\hat\rho_{\mathrm{in}}|+}, \qquad
p_{111} = |\beta|^2 = \braket{-|\hat\rho_{\mathrm{in}}|-} ,
$$
so $\hat{E}_{000} = \ket{+}\!\bra{+}$ and $\hat{E}_{111} = \ket{-}\!\bra{-}$.

**The span.** The operator space of one qubit is $4$-dimensional, spanned by
$\{\hat{I},\hat{X},\hat{Y},\hat{Z}\}$. An informationally complete POVM spans all of it. Counting
$\dim\mathrm{span}\{\hat{E}_b\}$ measures how many of the four coordinates of
$\hat\rho_{\mathrm{in}} = \tfrac12(\hat{I} + \vec{r}\cdot\vec{\sigma})$ the features can see.

## 3. Part 1: how many POVM elements?

The joint system has dimension $2\times4 = 8$, so a computational-basis measurement yields $8$
outcomes and $8$ effects $\hat{E}_b$. Only two of them are nonzero.

In [ ]:
import numpy as np, collections
from functools import reduce

np.set_printoptions(precision=4, suppress=True)

I2 = np.eye(2, dtype=complex)
X  = np.array([[0, 1], [1, 0]], dtype=complex)
Y  = np.array([[0, -1j], [1j, 0]])
Z  = np.array([[1, 0], [0, -1]], dtype=complex)
H  = np.array([[1, 1], [1, -1]], dtype=complex) / np.sqrt(2)

plus, minus = np.array([1, 1]) / np.sqrt(2), np.array([1, -1]) / np.sqrt(2)

def kron(*ops):
    return reduce(np.kron, ops)

def cnot(N, control, target):
    """CNOT on N qubits, 0-based indices, qubit 0 most significant."""
    M = np.zeros((2**N, 2**N), dtype=complex)
    for b in range(2**N):
        bits = [(b >> (N - 1 - k)) & 1 for k in range(N)]
        if bits[control]:
            bits[target] ^= 1
        M[sum(bit << (N - 1 - k) for k, bit in enumerate(bits)), b] = 1
    return M

# V = CNOT_23 CNOT_12 H_1, with qubit 0 the input and qubits 1, 2 the reservoir.
V = cnot(3, 1, 2) @ cnot(3, 0, 1) @ kron(H, I2, I2)
assert np.allclose(V.conj().T @ V, np.eye(8)), "V must be unitary"

def effects(W):
    """E_b = <00|_res W^dag |b><b| W |00>_res, an operator on the input qubit."""
    E = {}
    for b in range(8):
        A_b = np.array([W[b, (i << 2)] for i in (0, 1)])   # <b| W |i,00>
        E[format(b, '03b')] = np.outer(A_b.conj(), A_b)
    return E

E = effects(V)
nonzero = [b for b, M in E.items() if np.linalg.norm(M) > 1e-12]
print(f"outcomes available : {len(E)}")
print(f"nonzero effects    : {len(nonzero)}  ->  {nonzero}\n")
for b in sorted(E):
    if b in nonzero:
        print(f"  E_{b} =")
        print(f"{E[b].real}")
    else:
        print(f"  E_{b} = zero")

In [ ]:
print("E_000 == |+><+|              :", np.allclose(E['000'], np.outer(plus, plus)))
print("E_111 == |-><-|              :", np.allclose(E['111'], np.outer(minus, minus)))
print("other six vanish             :", all(np.allclose(E[b], 0) for b in E if b not in nonzero))
print("completeness  sum_b E_b == I :", np.allclose(sum(E.values()), I2))

assert np.allclose(E['000'], np.outer(plus, plus))
assert np.allclose(E['111'], np.outer(minus, minus))
assert all(np.allclose(E[b], 0) for b in E if b not in nonzero)
assert np.allclose(sum(E.values()), I2)

Eight in principle, two in practice. The CNOT chain copies the input's $\hat{X}$-basis value into every
qubit, so the three qubits always agree and only the aligned strings $000$ and $111$ can occur. The
larger reservoir supplies the extra outcomes, and this circuit leaves them empty.

## 4. Part 2: off-diagonal entries, and what they are worth

Both survivors do carry off-diagonal weight,
$$
\hat{E}_{000} = \ket{+}\!\bra{+} = \frac{1}{2}\begin{pmatrix}1&1\\1&1\end{pmatrix}, \qquad
\hat{E}_{111} = \ket{-}\!\bra{-} = \frac{1}{2}\begin{pmatrix}1&-1\\-1&1\end{pmatrix},
$$
so $|\braket{0|\hat{E}_b|1}| = \tfrac12$ for both, where the one-reservoir-qubit example of Section 7.5
gave strictly diagonal effects at $\theta$ fixed. That looks like progress and is not. Both operators
are projectors onto $\hat{X}$ eigenstates, so each commutes with $\hat{X}$, and the features
$$
f_{000} = \tfrac12\bigl(1 + r_x\bigr), \qquad f_{111} = \tfrac12\bigl(1 - r_x\bigr)
$$
depend on the input only through $\braket{\hat{X}} = r_x$. The Hadamard has rotated the single resolved
axis from $\hat{Z}$ to $\hat{X}$, nothing more. Off-diagonal entries in the computational basis record
which axis is being read, not how many.

In [ ]:
def povm_span(E, tol=1e-8):
    """Dimension of span{E_b} inside the 4-dimensional operator space of one qubit."""
    M = np.array([m.flatten() for m in E.values() if np.linalg.norm(m) > 1e-12])
    return np.linalg.matrix_rank(M, tol=tol)

for b in nonzero:
    print(f"  |<0|E_{b}|1>| = {abs(E[b][0, 1]):.4f}   [E_{b}, X] = 0 : "
          f"{np.allclose(E[b] @ X - X @ E[b], 0)}")
    assert np.isclose(abs(E[b][0, 1]), 0.5)
    assert np.allclose(E[b] @ X - X @ E[b], 0)

print(f"\ndim span{{E_b}} = {povm_span(E)} of 4  ->  spans only {{I, X}}")
assert povm_span(E) == 2

In [ ]:
def features(E, r):
    """f_b = Tr[E_b rho] for rho = (I + r.sigma)/2."""
    rho = (I2 + r[0] * X + r[1] * Y + r[2] * Z) / 2
    return {b: np.trace(m @ rho).real for b, m in E.items() if np.linalg.norm(m) > 1e-12}

rng = np.random.default_rng(0)
print(f"{'r_x':>8} {'r_y':>8} {'r_z':>8} | {'f_000':>9} {'(1+r_x)/2':>10} | {'f_111':>9} {'(1-r_x)/2':>10}")
for _ in range(6):
    r = rng.normal(size=3)
    r /= 1.3 * np.linalg.norm(r)                 # a valid Bloch vector, |r| < 1
    f = features(E, r)
    print(f"{r[0]:>8.4f} {r[1]:>8.4f} {r[2]:>8.4f} | {f['000']:>9.5f} {(1 + r[0]) / 2:>10.5f} "
          f"| {f['111']:>9.5f} {(1 - r[0]) / 2:>10.5f}")
    assert np.isclose(f['000'], (1 + r[0]) / 2)
    assert np.isclose(f['111'], (1 - r[0]) / 2)

print("\nBoth features track r_x exactly and ignore r_y, r_z.")

Two features that sum to one, so there is a single independent number in the readout. Any target
function of $r_y$ or $r_z$ is unlearnable here, whatever the training set.

### What local dressing can and cannot repair

The measurement is performed in the same basis the CNOTs copy into, which is what collapses the
readout. Rotating the input and the reservoir *together*, a quantum-eraser configuration, re-accesses
the coherence between the two copied branches. Dressing the circuit with a Hadamard on every qubit is
the cleanest case, and turns the apparatus into a $\hat{Z}$ readout.

In [ ]:
E_had = effects(kron(H, H, H) @ V)

print("Hadamard on every qubit after the entangler:\n")
print(f"{'b':>6} {'parity':>7} {'E_b':>18}")
for b in sorted(E_had):
    parity = sum(int(c) for c in b) % 2
    target = np.outer([1, 0], [1, 0]) / 4 if parity == 0 else np.outer([0, 1], [0, 1]) / 4
    label = '|0><0|/4' if parity == 0 else '|1><1|/4'
    assert np.allclose(E_had[b], target)
    print(f"{b:>6} {parity:>7} {label:>18}")

print(f"\ncompleteness : {np.allclose(sum(E_had.values()), I2)}")
print(f"dim span     : {povm_span(E_had)} of 4  ->  spans {{I, Z}}, a Z readout")
assert np.allclose(sum(E_had.values()), I2)
assert povm_span(E_had) == 2

All eight outcomes are now populated and every effect is diagonal, the mirror image of where we
started. The axis moved from $\hat{X}$ to $\hat{Z}$ and the count of resolved components did not
change.

Generic local dressings do better, though only by one. Write $\hat{W} = \hat{w}_1\otimes\hat{w}_2
\otimes\hat{w}_3$ applied after $\hat{V}$, and let $c_b = \bra{b}\hat{W}\ket{000}$,
$d_b = \bra{b}\hat{W}\ket{111}$. In the $\ket{\pm}$ basis the effect reads
$$
\hat{E}_b = |c_b|^2\,\ket{+}\!\bra{+} + |d_b|^2\,\ket{-}\!\bra{-}
 + \bigl(c_b d_b^{*}\,\ket{+}\!\bra{-} + \mathrm{h.c.}\bigr) .
$$
The off-diagonal factorizes, $c_b d_b^{*} = \prod_j (\hat{w}_j)_{b_j 0}(\hat{w}_j)^{*}_{b_j 1}$, and
orthogonality of the columns of each $\hat{w}_j$ forces
$(\hat{w}_j)_{00}(\hat{w}_j)^{*}_{01} = -(\hat{w}_j)_{10}(\hat{w}_j)^{*}_{11}$. Every factor therefore
only changes sign with $b_j$, giving
$$
c_b d_b^{*} = (-1)^{|b|}\,\zeta , \qquad \zeta = \prod_j (\hat{w}_j)_{00}(\hat{w}_j)^{*}_{01} ,
$$
a single complex number up to sign. All eight effects share one fixed off-diagonal direction, so
$$
\mathrm{span}\{\hat{E}_b\} \subseteq \mathrm{span}\{\ket{+}\!\bra{+},\, \ket{-}\!\bra{-},\, \hat{M}\},
\qquad \hat{M} = \zeta\ket{+}\!\bra{-} + \zeta^{*}\ket{-}\!\bra{+} ,
$$
of dimension at most $3$. The first two span $\{\hat{I},\hat{X}\}$ and $\hat{M}$ is one fixed direction
in the $\hat{Y}$-$\hat{Z}$ plane. Generic local dressings resolve two independent Bloch components, and
the third is out of reach for every one of them.

In [ ]:
def haar_u1(rng):
    """Haar-random U(2): QR of a Ginibre matrix with the R-diagonal phase fix (Ch. 1)."""
    A = (rng.normal(size=(2, 2)) + 1j * rng.normal(size=(2, 2))) / np.sqrt(2)
    Q, R = np.linalg.qr(A)
    return Q * (np.diag(R) / np.abs(np.diag(R)))

def tally(counter):
    """Readable histogram of the observed span dimensions."""
    return {int(k): int(v) for k, v in sorted(counter.items())}

rng = np.random.default_rng(7)
counts_post = collections.Counter()
counts_both = collections.Counter()
for _ in range(300):
    W   = kron(haar_u1(rng), haar_u1(rng), haar_u1(rng))     # dressing after the entangler
    pre = kron(haar_u1(rng), I2, I2)                         # plus a rotation of the input
    counts_post[povm_span(effects(W @ V))] += 1
    counts_both[povm_span(effects(W @ V @ pre))] += 1

print("dim span{E_b} over 300 random local dressings")
print(f"  dressing after V           : {tally(counts_post)}")
print(f"  dressing after V + pre-rot : {tally(counts_both)}")
assert set(counts_post) == {3} and set(counts_both) == {3}
print("\nAlways 3, never 4. Local dressing of a perfect one-axis copy is never")
print("informationally complete.")

The obstruction is the copy itself. The CNOT chain writes one input axis into three qubits and
discards the rest, and no local operation afterwards can recover what the entangler threw away. What
does work is an entangler that does not perform a perfect copy in the first place. The partial-strength
coupling $e^{-i\theta\hat{X}_{\mathrm{in}}\hat{X}_{\mathrm{res}}}$ of Section 7.5, dressed with
single-qubit rotations, leaves the branches only partly correlated and reaches all four operator
directions.

In [ ]:
def exp_xx(N, a, b, theta):
    """exp(-i theta X_a X_b) on N qubits."""
    XX = kron(*[X if k in (a, b) else I2 for k in range(N)])
    return np.cos(theta) * np.eye(2**N) - 1j * np.sin(theta) * XX

V_partial = exp_xx(3, 1, 2, 0.7) @ exp_xx(3, 0, 1, 0.9)     # never a perfect copy

counts = collections.Counter()
for _ in range(300):
    W   = kron(haar_u1(rng), haar_u1(rng), haar_u1(rng))
    pre = kron(haar_u1(rng), I2, I2)
    counts[povm_span(effects(W @ V_partial @ pre))] += 1

print(f"CNOT chain              : dim span{{E_b}} = {tally(counts_both)}   (r_x, r_y, r_z: 2 of 3)")
print(f"exp(-i theta X X) chain : dim span{{E_b}} = {tally(counts)}   (informationally complete)")
assert set(counts) == {4}

## 5. What the reservoir actually bought

| circuit | outcomes | nonzero $\hat{E}_b$ | $\dim\mathrm{span}\{\hat{E}_b\}$ | resolved |
|---|---|---|---|---|
| $\hat{V} = \mathrm{CNOT}_{23}\mathrm{CNOT}_{12}\hat{H}_1$ | $8$ | $2$ | $2$ | $r_x$ |
| $\hat{V}$ dressed with $\hat{H}^{\otimes3}$ | $8$ | $8$ | $2$ | $r_z$ |
| $\hat{V}$ dressed generically | $8$ | $8$ | $3$ | two components |
| $e^{-i\theta\hat{X}\hat{X}}$ chain, dressed | $8$ | $8$ | $4$ | all three |

Reading down the second and fourth columns, the number of outcomes is constant at $8$ and the number
of independent operators runs from $2$ to $4$. Outcomes are free and independence is not, which is the
whole content of the exercise.

## Conclusion

Two reservoir qubits offer $8$ POVM elements on the input qubit, and this circuit populates $2$ of
them, $\hat{E}_{000} = \ket{+}\!\bra{+}$ and $\hat{E}_{111} = \ket{-}\!\bra{-}$. They are complete,
they have off-diagonal entries of magnitude $\tfrac12$, and they commute with $\hat{X}$, so the
readout is a refined $\hat{X}$ measurement and the features see $r_x$ alone. Adding reservoir qubits
enlarged the dilation space without enlarging the span, because the CNOT chain copies a single axis
and the measurement is made in the basis it copies into.

Local dressing rotates the resolved axis and, generically, buys one more component, capping at $3$ of
$4$. Informational completeness needs an entangler that does not reduce to a one-axis copy, which is
the partial-strength coupling of Section 7.5 or the generic dynamics of a scrambling reservoir. The
useful diagnostic for a QELM is therefore $\dim\mathrm{span}\{\hat{E}_b\}$, and never the outcome
count. Section 7.5 supplies the other half of the story, since a fully scrambling reservoir spans the
operator space but drives every $\hat{E}_b$ toward $\hat{I}_{\mathrm{in}}/D$, and the working regime
sits between the copy and the Haar limit.